In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train = X[X["moon"].isin(splits["train"])]
y_train = y[y["moon"].isin(splits["train"])]

X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import time
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import spearmanr, pearsonr, rankdata
from sklearn.linear_model import ElasticNet


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    t_start = time.perf_counter()

    # ── Config ────────────────────────────────────────────────────────────────
    VAL_WINDOW = 15   # larger validation window for more stable estimates

    GRID = {
        "lookback": [50, 100, 150, 200, 300],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "alpha":    [0.0001, 0.0005, 0.001, 0.005],
    }

    # ── Utilities ──────────────────────────────────────────────────────────────
    def cs_rank_series(s):
        n = len(s)
        if n < 2: return s * 0.0
        r = s.rank(method="average")
        return (r - 1) / (n - 1) - 0.5

    def fast_cs_rank(df, cols):
        df = df.copy()
        out = np.empty((len(df), len(cols)), dtype=np.float32)
        for _, idx in df.groupby("moon", sort=False).groups.items():
            ia = idx.values
            block = df.iloc[ia][cols].values.astype(np.float32)
            n = len(ia)
            if n < 2:
                out[ia] = 0.0
                continue
            ranks = np.argsort(np.argsort(block, axis=0), axis=0).astype(np.float32)
            out[ia] = ranks / (n - 1) - 0.5
        for i, c in enumerate(cols):
            df[c] = out[:, i]
        return df

    def fill_median(df, cols):
        df = df.copy()
        medians = df.groupby("moon")[cols].transform("median")
        for c in cols:
            mask = df[c].isna()
            df.loc[mask, c] = medians.loc[mask, c]
        df[cols] = df[cols].fillna(0.0)
        return df

    def moon_spearman(yt, yp):
        if len(yt) < 5: return 0.0
        if np.unique(yt).size < 2 or np.unique(yp).size < 2:
            return 0.0
        r, _ = spearmanr(yp, yt)
        return float(r) if not np.isnan(r) else 0.0

    # ── Prepare ────────────────────────────────────────────────────────────────
    print("  [1/4] Preparing data ...")
    df = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"])
    df = df.dropna(subset=["target"])
    
    ALL_FEATS = [c for c in df.columns if c.startswith("Feature_") and df[c].nunique() > 1]
    print(f"  Using {len(ALL_FEATS)} feature columns")
    
    df = fill_median(df, ALL_FEATS)
    df = fast_cs_rank(df, ALL_FEATS)
    df["target_rank"] = df.groupby("moon")["target"].transform(cs_rank_series)
    all_moons = sorted(df["moon"].unique())
    print(f"  Moons: {len(all_moons)}  ({all_moons[0]}→{all_moons[-1]})")

    val_moons = all_moons[-VAL_WINDOW:]

    X_arr      = df[ALL_FEATS].values.astype(np.float32)
    y_rank_arr = df["target_rank"].values.astype(np.float32)
    moon_arr   = df["moon"].values
    y_raw_arr  = df["target"].values.astype(np.float32)

    combos = list(product(GRID["lookback"], GRID["l1_ratio"], GRID["alpha"]))
    print(f"  [2/4] Grid search: {len(combos)} combos ({len(GRID['lookback'])} lookbacks x "
          f"{len(GRID['l1_ratio'])} l1_ratios x {len(GRID['alpha'])} alphas)")

    best_score, best_params = -np.inf, {}
    for i, (lb, l1, al) in enumerate(combos, 1):
        train_avail = [m for m in all_moons if m < val_moons[0]]
        tr_moons    = train_avail[-lb:] if lb < len(train_avail) else train_avail
        if len(tr_moons) < 5:
            continue
        tr_mask  = np.isin(moon_arr, tr_moons)
        val_mask = np.isin(moon_arr, val_moons)
        try:
            m = ElasticNet(alpha=al, l1_ratio=l1,
                           max_iter=2000, random_state=42, fit_intercept=False)
            m.fit(X_arr[tr_mask], y_rank_arr[tr_mask])
            preds = m.predict(X_arr[val_mask])
        except Exception:
            continue
        yval     = y_raw_arr[val_mask]
        moon_val = moon_arr[val_mask]
        ics = [moon_spearman(yval[moon_val==mv], preds[moon_val==mv])
               for mv in val_moons if (moon_val==mv).sum() >= 5]
        score = float(np.mean(ics)) if ics else -999.0
        if score > best_score:
            best_score  = score
            best_params = {"lookback": lb, "l1_ratio": l1, "alpha": al}
            elapsed = time.perf_counter() - t_start
            print(f"    [{i:4d}/{len(combos)}] lb={lb:4d} l1={l1:.1f} a={al:.5f} "
                  f"CV={score:+.5f}  [{elapsed:.0f}s]")

    print(f"  Best CV Spearman={best_score:+.5f}  params={best_params}")

    # ── Refit ElasticNet on optimal window ────────────────────────────────────
    lb = best_params["lookback"]
    fit_moons = all_moons[-lb:] if lb < len(all_moons) else all_moons
    df_fit    = df[df["moon"].isin(fit_moons)]
    X_fit     = df_fit[ALL_FEATS].values.astype(np.float32)
    y_fit     = df_fit["target_rank"].values

    print(f"  [3/4] ElasticNet fit: {len(fit_moons)} moons, {len(df_fit):,} rows ...")
    enet = ElasticNet(alpha=best_params["alpha"], l1_ratio=best_params["l1_ratio"],
                      max_iter=5000, random_state=42, fit_intercept=False)
    enet.fit(X_fit, y_fit)
    nz = [(f, round(c,6)) for f,c in zip(ALL_FEATS, enet.coef_) if abs(c)>1e-7]
    print(f"  Non-zero coefs: {len(nz)}/{len(ALL_FEATS)}")
    for fn, cv in sorted(nz, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f"    {fn}: {cv:+.6f}")

    # ── Save model ─────────────────────────────────────────────────────────────
    print("  [4/4] Saving model ...")
    joblib.dump({
        "enet": enet,
        "all_feats": ALL_FEATS,
        "best_params": best_params,
        "best_cv": best_score,
    }, os.path.join(model_directory_path, "model.joblib"))

    print(f"  Done in {time.perf_counter()-t_start:.1f}s | saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    feats    = obj["all_feats"]
    enet     = obj["enet"]

    X = X_test.copy()

    # Fill NaN with cross-sectional median
    for c in feats:
        if c in X.columns:
            med = X[c].median()
            X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)

    # Cross-sectional rank per feature
    feat_arr = X[feats].values.astype(np.float32)
    n = len(feat_arr)
    ranks = np.argsort(np.argsort(feat_arr, axis=0), axis=0).astype(np.float32)
    feat_arr = ranks / max(n - 1, 1) - 0.5

    # ElasticNet prediction
    preds_raw = enet.predict(feat_arr)
    final = (rankdata(preds_raw) - 1) / max(n - 1, 1) - 0.5

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": final,
    })


In [ ]:
embargo = 4
model_dir = "./model_cr3"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    X_moon = X_test_cloud[X_test_cloud["moon"] == moon]
    pred = infer(X_moon, model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

from scipy.stats import pearsonr

for moon in splits["reduced_cloud"]:
    p  = predictions[predictions["moon"] == moon]
    gt = y_test_cloud[y_test_cloud["moon"] == moon]
    merged = p.merge(gt, on=["moon", "id"])
    if len(merged) > 1:
        corr, _ = pearsonr(merged["prediction"], merged["target"])
        print(f"Moon {moon}: Pearson r = {corr:.4f}")
